# Experiment : Structural Parameter Recovery

**Question:** Can self-attention recover the generator's item×item interaction
matrix $C$ from choice data alone?

All previous experiments measured *behavioural* recovery — does the model
reproduce the planted effect. This experiment measures *structural* recovery —
do the model's learned pre-softmax scores $q_i \cdot k_j$ match $C_{ij}$ itself.

## Cell 1: Item×item CDM generator with rank control

We move from the feature×feature formulation to the item×item CDM of
Seshadri et al. Each item $i$ has a baseline utility $u_i$. When item $j$ is
present in the choice set $S$, it shifts item $i$'s utility by $C_{ij}$:

$$
\text{score}(i \mid S) = u_i + \sum_{j \in S,\, j \neq i} C_{ij}
$$

Choice probabilities are then softmax over scores within $S$.

**Rank control.** We build $C = AB^\top$ with $A, B \in \mathbb{R}^{N \times r}$,
giving a dense matrix of rank exactly $r$. Since attention computes
$q_i \cdot k_j$ with head dimension $d$, its induced score matrix has rank at
most $d$ — so $r$ is the dial that should determine whether recovery is even
possible.

**Diagonal.** $C_{ii}$ is never used (an item does not pull on itself), so we
zero it out and exclude it from all recovery comparisons.

In [ ]:
import numpy as np
import torch

def make_C(N, r, scale=1.0, seed=0):
    """Rank-r item×item interaction matrix, C = A @ B.T, diagonal zeroed."""
    rng = np.random.default_rng(seed)
    A = rng.normal(0, 1, size=(N, r))
    B = rng.normal(0, 1, size=(N, r))
    C = A @ B.T
    C = C / np.abs(C).mean() * scale   # normalise effect size
    np.fill_diagonal(C, 0.0)
    return C

def make_utilities(N, scale=1.0, seed=0):
    rng = np.random.default_rng(seed + 999)
    return rng.normal(0, scale, size=N)

def generate_choices(N, C, u, n_samples, set_size, seed=0):
    """Sample choice sets uniformly, then choices via CDM softmax."""
    rng = np.random.default_rng(seed + 1234)
    sets    = np.zeros((n_samples, set_size), dtype=np.int64)
    choices = np.zeros(n_samples, dtype=np.int64)

    for n in range(n_samples):
        S = rng.choice(N, size=set_size, replace=False)
        # pull on each item from all others in S (diagonal is 0, so self-term vanishes)
        scores = u[S] + C[np.ix_(S, S)].sum(axis=1)
        p = np.exp(scores - scores.max())
        p /= p.sum()
        sets[n]    = S
        choices[n] = rng.choice(set_size, p=p)   # index *within* the set

    return torch.tensor(sets), torch.tensor(choices)


# --- smoke test ---
N, r, set_size, n_samples = 30, 4, 6, 20000

C = make_C(N, r, scale=0.8, seed=0)
u = make_utilities(N, scale=1.0, seed=0)
sets, choices = generate_choices(N, C, u, n_samples, set_size, seed=0)

print(f"C shape: {C.shape}, rank: {np.linalg.matrix_rank(C)} (target r={r})")
print(f"C off-diagonal: mean={C[~np.eye(N,dtype=bool)].mean():.3f}, "
      f"std={C[~np.eye(N,dtype=bool)].std():.3f}")
print(f"sets: {tuple(sets.shape)}, choices: {tuple(choices.shape)}")
print(f"choice index distribution: {np.bincount(choices.numpy(), minlength=set_size)}")

C shape: (30, 30), rank: 30 (target r=4)
C off-diagonal: mean=-0.036, std=1.023
sets: (20000, 6), choices: (20000,)
choice index distribution: [3351 3410 3242 3374 3300 3323]


### Cell 1 (corrected): diagonal handling

Zeroing the diagonal of a rank-$r$ matrix is a rank-$N$ perturbation, which
destroys the rank structure. Instead we leave $C = AB^\top$ untouched at rank
exactly $r$, and subtract the self-term explicitly when scoring:

$$
\text{score}(i \mid S) = u_i + \sum_{j \in S} C_{ij} - C_{ii}
$$

**Consequence.** $C_{ii}$ now never influences the data, so it is
unidentifiable — no model can recover it, and it must be excluded from all
recovery comparisons. This costs us nothing: attention's own $q_i \cdot k_i$
is equally free. If $d \geq r$, a rank-$d$ matrix matching $C$ off-diagonal
exists (namely $C$ itself), so the capacity argument holds unchanged.

Note the boundary may be slightly soft just below $r = d$: the free diagonal
gives the model a little slack to fit the off-diagonal with lower rank than
$C$ itself.

In [ ]:
def make_C(N, r, scale=1.0, seed=0):
    """Rank-r item×item interaction matrix, C = A @ B.T. Diagonal left intact."""
    rng = np.random.default_rng(seed)
    A = rng.normal(0, 1, size=(N, r))
    B = rng.normal(0, 1, size=(N, r))
    C = A @ B.T
    off = ~np.eye(N, dtype=bool)
    C = C / np.abs(C[off]).mean() * scale   # normalise on off-diagonal only
    return C

def generate_choices(N, C, u, n_samples, set_size, seed=0):
    rng = np.random.default_rng(seed + 1234)
    sets    = np.zeros((n_samples, set_size), dtype=np.int64)
    choices = np.zeros(n_samples, dtype=np.int64)

    for n in range(n_samples):
        S = rng.choice(N, size=set_size, replace=False)
        C_sub  = C[np.ix_(S, S)]
        scores = u[S] + C_sub.sum(axis=1) - np.diag(C_sub)   # exclude self-pull
        p = np.exp(scores - scores.max())
        p /= p.sum()
        sets[n]    = S
        choices[n] = rng.choice(set_size, p=p)

    return torch.tensor(sets), torch.tensor(choices)


# --- smoke test ---
N, r, set_size, n_samples = 30, 4, 6, 20000

C = make_C(N, r, scale=0.8, seed=0)
u = make_utilities(N, scale=1.0, seed=0)
sets, choices = generate_choices(N, C, u, n_samples, set_size, seed=0)

off = ~np.eye(N, dtype=bool)
ctx_spread = (C[np.ix_(sets[0].numpy(), sets[0].numpy())].sum(axis=1)).std()

print(f"C rank: {np.linalg.matrix_rank(C)} (target r={r})")
print(f"C off-diag: mean={C[off].mean():.3f}, std={C[off].std():.3f}")
print(f"u std: {u.std():.3f}   |   context-term spread (one set): {ctx_spread:.3f}")
print(f"choice index distribution: {np.bincount(choices.numpy(), minlength=set_size)}")

C rank: 4 (target r=4)
C off-diag: mean=-0.036, std=1.031
u std: 0.918   |   context-term spread (one set): 2.165
choice index distribution: [3354 3416 3227 3379 3306 3318]


## Cell 2: Attention model + structural recovery metric

**Model.** Three embedding tables, no features, no shared projections:

- $q_i, k_i \in \mathbb{R}^d$ — query and key, per item
- $u_i \in \mathbb{R}$ — baseline utility, per item

Score for item $i$ in slate $S$ (self-term excluded, matching the generator):

$$
\text{score}(i \mid S) = u_i + \sum_{j \in S,\, j \neq i} q_i \cdot k_j
$$

Note this is the **no-softmax-inside** variant: pulls are summed raw, exactly
as CDM does. This is rung 3 of the architecture ladder and the most generous
starting point. Vanilla softmax attention comes later as a comparison.

**Recovery metric.** Two nuisance degrees of freedom must be removed before
comparing $\hat{C}$ to $C$:

1. *Row/column shifts.* Adding a constant to row $i$ of $C$ is absorbed by
   $u_i$; adding a constant to column $j$ shifts every item in any slate
   containing $j$ equally, and softmax over the slate cancels it. Neither is
   identifiable, so we double-center both matrices.
2. *Global scale.* We report Pearson correlation, which is scale-free.

The diagonal is excluded throughout — it never influences the data.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ItemAttention(nn.Module):
    def __init__(self, N, d):
        super().__init__()
        self.q = nn.Embedding(N, d)
        self.k = nn.Embedding(N, d)
        self.u = nn.Embedding(N, 1)
        for emb in (self.q, self.k):
            nn.init.normal_(emb.weight, std=0.1)
        nn.init.zeros_(self.u.weight)

    def forward(self, sets):                      # sets: (B, M)
        Q = self.q(sets)                          # (B, M, d)
        K = self.k(sets)                          # (B, M, d)
        pulls = torch.bmm(Q, K.transpose(1, 2))   # (B, M, M)
        self_pull = torch.diagonal(pulls, dim1=1, dim2=2)
        ctx = pulls.sum(dim=2) - self_pull         # exclude self-term
        return self.u(sets).squeeze(-1) + ctx      # (B, M) logits

    def implied_C(self):
        with torch.no_grad():
            return (self.q.weight @ self.k.weight.T).cpu().numpy()


def double_center(M):
    """Remove row and column shifts (both unidentifiable)."""
    M = M - M.mean(axis=1, keepdims=True)
    M = M - M.mean(axis=0, keepdims=True)
    return M

def recovery_score(C_true, C_hat):
    """Pearson correlation on double-centered off-diagonal entries."""
    N = C_true.shape[0]
    off = ~np.eye(N, dtype=bool)
    a = double_center(C_true)[off]
    b = double_center(C_hat)[off]
    return float(np.corrcoef(a, b)[0, 1])


def train_model(sets, choices, N, d, epochs=300, lr=0.05, seed=0, verbose=True):
    torch.manual_seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = ItemAttention(N, d).to(dev)
    sets, choices = sets.to(dev), choices.to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(model(sets), choices)
        loss.backward()
        opt.step()
        if verbose and (ep + 1) % 100 == 0:
            print(f"  epoch {ep+1:4d}  NLL {loss.item():.4f}")
    return model, loss.item()


# --- sanity check: does the metric work on a known-good matrix? ---
print("Metric sanity checks")
print(f"  C vs C            : {recovery_score(C, C):.4f}   (expect 1.0)")
print(f"  C vs C + row shift: {recovery_score(C, C + np.arange(N)[:, None]):.4f}   (expect 1.0)")
print(f"  C vs random       : {recovery_score(C, np.random.randn(N, N)):.4f}   (expect ~0)")

# --- first real run: d = r = 4, the easiest possible case ---
print(f"\nTraining  N={N}  r={r}  d=4")
model, final_nll = train_model(sets, choices, N, d=4, epochs=300)

print(f"\nfinal NLL   : {final_nll:.4f}   (uniform baseline = {np.log(set_size):.4f})")
print(f"recovery    : {recovery_score(C, model.implied_C()):.4f}")

Metric sanity checks
  C vs C            : 1.0000   (expect 1.0)
  C vs C + row shift: 1.0000   (expect 1.0)
  C vs random       : -0.0016   (expect ~0)

Training  N=30  r=4  d=4
  epoch  100  NLL 0.8712
  epoch  200  NLL 0.8703
  epoch  300  NLL 0.8702

final NLL   : 0.8702   (uniform baseline = 1.7918)
recovery    : 0.9950


## Cell 3: Rank × head-dimension sweep

For each true rank $r$ and head dimension $d$, we regenerate data, train from
scratch, and record recovery. The prediction:

- $d \geq r$ → recovery high (capacity sufficient)
- $d < r$ → recovery collapses (capacity ceiling binds)

so the grid should show a **boundary along the diagonal**. Anything else
falsifies the rank argument.

Two controls matter here:

**Fresh data per $r$.** A rank-8 $C$ is a different generator from a rank-4 $C$,
so data must be regenerated, not reused across the $r$ axis.

**Effect size held constant.** `make_C` normalises on the off-diagonal mean, so
context strength stays comparable as $r$ varies. Without this, higher-rank
matrices would also be *stronger*, confounding rank with signal magnitude.

We run 3 seeds per cell and report mean ± std, since single runs at the
boundary will be noisy.

In [ ]:
import pandas as pd
import time

r_values = [1, 2, 4, 8, 16, 30]
d_values = [1, 2, 4, 8, 16, 30]
seeds    = [0, 1, 2]

rows = []
t0 = time.time()

for r_true in r_values:
    for s in seeds:
        C_s = make_C(N, r_true, scale=0.8, seed=100 * s + r_true)
        u_s = make_utilities(N, scale=1.0, seed=100 * s + r_true)
        sets_s, ch_s = generate_choices(N, C_s, u_s, n_samples, set_size,
                                        seed=100 * s + r_true)
        for d_head in d_values:
            m, nll = train_model(sets_s, ch_s, N, d=d_head,
                                 epochs=300, seed=s, verbose=False)
            rows.append({'r': r_true, 'd': d_head, 'seed': s,
                         'recovery': recovery_score(C_s, m.implied_C()),
                         'nll': nll})
    print(f"r={r_true:2d} done  ({time.time()-t0:.0f}s)")

df = pd.DataFrame(rows)
df.to_csv('exp4_rank_sweep.csv', index=False)

piv_m = df.pivot_table(index='r', columns='d', values='recovery', aggfunc='mean')
piv_s = df.pivot_table(index='r', columns='d', values='recovery', aggfunc='std')

print("\nRECOVERY  (rows = true rank r, cols = head dim d)\n")
print(piv_m.round(3).to_string())
print("\nSTD ACROSS SEEDS\n")
print(piv_s.round(3).to_string())

print("\nNLL  (lower = better fit)\n")
print(df.pivot_table(index='r', columns='d', values='nll',
                     aggfunc='mean').round(3).to_string())

r= 1 done  (18s)
r= 2 done  (37s)
r= 4 done  (55s)
r= 8 done  (74s)
r=16 done  (92s)
r=30 done  (112s)

RECOVERY  (rows = true rank r, cols = head dim d)

d      1      2      4      8      16     30
r                                           
1   0.999  0.998  0.996  0.993  0.990  0.988
2   0.786  0.998  0.994  0.988  0.982  0.980
4   0.667  0.842  0.997  0.992  0.987  0.985
8   0.539  0.710  0.887  0.994  0.988  0.986
16  0.470  0.610  0.776  0.922  0.985  0.981
30  0.407  0.557  0.720  0.874  0.969  0.974

STD ACROSS SEEDS

d      1      2      4      8      16     30
r                                           
1   0.001  0.000  0.000  0.001  0.001  0.002
2   0.104  0.001  0.001  0.002  0.003  0.004
4   0.049  0.013  0.001  0.001  0.001  0.002
8   0.005  0.032  0.019  0.000  0.001  0.002
16  0.038  0.029  0.015  0.005  0.003  0.003
30  0.012  0.006  0.021  0.011  0.007  0.010

NLL  (lower = better fit)

d      1      2      4      8      16     30
r                                

## Cell 4: Is attention optimal given its rank constraint?

A rank-$d$ score matrix can at best match the top-$d$ singular directions of
$C$. By Eckart–Young the achievable correlation is

$$
\text{ceiling}(d) = \sqrt{\frac{\sum_{i \leq d} \sigma_i^2}{\sum_i \sigma_i^2}}
$$

on the double-centered $C$. If observed recovery ≈ ceiling, attention is not
*failing to learn* — it is learning the best rank-$d$ approximation available,
and the shortfall is purely representational.

The model has slight extra slack (free diagonal; centering can add up to two
effective dimensions), so small positive gaps are expected.

In [ ]:
def spectral_ceiling(C_true, d):
    M = double_center(C_true)
    s = np.linalg.svd(M, compute_uv=False)
    return float(np.sqrt((s[:d]**2).sum() / (s**2).sum()))

rows_c = []
for r_true in r_values:
    for s_ in seeds:
        C_s = make_C(N, r_true, scale=0.8, seed=100 * s_ + r_true)
        for d_head in d_values:
            rows_c.append({'r': r_true, 'd': d_head, 'seed': s_,
                           'ceiling': spectral_ceiling(C_s, d_head)})

cmp = df.merge(pd.DataFrame(rows_c), on=['r', 'd', 'seed'])
cmp['gap'] = cmp['recovery'] - cmp['ceiling']

print("SPECTRAL CEILING  (best possible at rank d)\n")
print(cmp.pivot_table(index='r', columns='d', values='ceiling',
                      aggfunc='mean').round(3).to_string())
print("\nGAP  (observed - ceiling;  ~0 = attention achieves its own limit)\n")
print(cmp.pivot_table(index='r', columns='d', values='gap',
                      aggfunc='mean').round(3).to_string())

SPECTRAL CEILING  (best possible at rank d)

d      1      2      4      8      16   30
r                                         
1   1.000  1.000  1.000  1.000  1.000  1.0
2   0.799  1.000  1.000  1.000  1.000  1.0
4   0.689  0.855  1.000  1.000  1.000  1.0
8   0.566  0.727  0.899  1.000  1.000  1.0
16  0.495  0.642  0.807  0.943  1.000  1.0
30  0.431  0.584  0.750  0.902  0.988  1.0

GAP  (observed - ceiling;  ~0 = attention achieves its own limit)

d      1      2      4      8      16     30
r                                           
1  -0.001 -0.002 -0.004 -0.007 -0.010 -0.012
2  -0.013 -0.002 -0.006 -0.012 -0.018 -0.020
4  -0.022 -0.013 -0.003 -0.008 -0.013 -0.015
8  -0.027 -0.017 -0.012 -0.006 -0.012 -0.014
16 -0.025 -0.032 -0.031 -0.021 -0.015 -0.019
30 -0.023 -0.026 -0.030 -0.027 -0.020 -0.026


## Cell 5: Softmax variant + held-out evaluation

Two changes.

**Held-out split.** All previous NLL numbers were training loss on the full
dataset. We now split 80/20 and report test NLL, so any prediction-vs-recovery
claim is not contaminated by overfitting.

**Softmax variant.** The current model sums pulls raw, matching CDM's additive
structure:

$$
\text{score}(i \mid S) = u_i + \sum_{j \neq i} q_i \cdot k_j
$$

Standard attention instead normalises the pulls so they sum to one across the
slate, then aggregates a learned value:

$$
\text{score}(i \mid S) = u_i + \sum_{j} \alpha_{ij} v_j,
\qquad \alpha_{ij} = \text{softmax}_j\!\left(\frac{q_i \cdot k_j}{\sqrt{d}}\right)
$$

This is zero-sum: more weight on one item means less on another. CDM's pulls
are not zero-sum. We test whether that mismatch is what produces the *silent
failure* seen in Experiment 2 — good predictive fit alongside poor structural
recovery.

The recovery metric is unchanged: we read the pre-softmax $q_i \cdot k_j$
matrix out of the trained model either way.

In [ ]:
class ItemAttentionSoftmax(nn.Module):
    """Standard attention: normalised weights over a learned value vector."""
    def __init__(self, N, d):
        super().__init__()
        self.q = nn.Embedding(N, d)
        self.k = nn.Embedding(N, d)
        self.v = nn.Embedding(N, d)
        self.u = nn.Embedding(N, 1)
        self.out = nn.Linear(d, 1)
        for emb in (self.q, self.k, self.v):
            nn.init.normal_(emb.weight, std=0.1)
        nn.init.zeros_(self.u.weight)
        self.d = d

    def forward(self, sets):
        Q, K, V = self.q(sets), self.k(sets), self.v(sets)
        logits = torch.bmm(Q, K.transpose(1, 2)) / np.sqrt(self.d)
        eye = torch.eye(sets.shape[1], device=sets.device, dtype=torch.bool)
        logits = logits.masked_fill(eye, -1e9)          # exclude self
        ctx = torch.bmm(F.softmax(logits, dim=2), V)     # (B, M, d)
        return self.u(sets).squeeze(-1) + self.out(ctx).squeeze(-1)

    def implied_C(self):
        with torch.no_grad():
            return (self.q.weight @ self.k.weight.T).cpu().numpy()


def train_eval(model_cls, sets, choices, N, d, epochs=300, lr=0.05,
               seed=0, split=0.8):
    torch.manual_seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    n_tr = int(split * len(sets))
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(sets), generator=g)
    tr, te = perm[:n_tr], perm[n_tr:]
    Xtr, ytr = sets[tr].to(dev), choices[tr].to(dev)
    Xte, yte = sets[te].to(dev), choices[te].to(dev)

    model = model_cls(N, d).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        F.cross_entropy(model(Xtr), ytr).backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        tr_nll = F.cross_entropy(model(Xtr), ytr).item()
        te_nll = F.cross_entropy(model(Xte), yte).item()
    return model, tr_nll, te_nll


rows2 = []
t0 = time.time()
for r_true in r_values:
    for s_ in seeds:
        C_s = make_C(N, r_true, scale=0.8, seed=100 * s_ + r_true)
        u_s = make_utilities(N, scale=1.0, seed=100 * s_ + r_true)
        X, y = generate_choices(N, C_s, u_s, n_samples, set_size,
                                seed=100 * s_ + r_true)
        for d_head in d_values:
            for name, cls in [('raw', ItemAttention),
                              ('softmax', ItemAttentionSoftmax)]:
                m, tr_n, te_n = train_eval(cls, X, y, N, d_head, seed=s_)
                rows2.append({'model': name, 'r': r_true, 'd': d_head,
                              'seed': s_, 'recovery': recovery_score(C_s, m.implied_C()),
                              'train_nll': tr_n, 'test_nll': te_n})
    print(f"r={r_true:2d} done  ({time.time()-t0:.0f}s)")

df2 = pd.DataFrame(rows2)
df2.to_csv('exp4_softmax_compare.csv', index=False)

for name in ['raw', 'softmax']:
    sub = df2[df2.model == name]
    print(f"\n{'='*52}\n{name.upper()}\n{'='*52}")
    print("\nRECOVERY\n")
    print(sub.pivot_table(index='r', columns='d', values='recovery',
                          aggfunc='mean').round(3).to_string())
    print("\nTEST NLL\n")
    print(sub.pivot_table(index='r', columns='d', values='test_nll',
                          aggfunc='mean').round(3).to_string())

print(f"\n{'='*52}\nSOFTMAX minus RAW\n{'='*52}")
piv = df2.pivot_table(index=['r', 'd'], columns='model',
                      values=['recovery', 'test_nll'], aggfunc='mean')
delta = pd.DataFrame({
    'd_recovery': piv[('recovery', 'softmax')] - piv[('recovery', 'raw')],
    'd_test_nll': piv[('test_nll', 'softmax')] - piv[('test_nll', 'raw')],
}).reset_index()
print("\nRECOVERY DIFF  (negative = softmax recovers worse)\n")
print(delta.pivot(index='r', columns='d', values='d_recovery').round(3).to_string())
print("\nTEST NLL DIFF  (negative = softmax predicts better)\n")
print(delta.pivot(index='r', columns='d', values='d_test_nll').round(3).to_string())

r= 1 done  (39s)
r= 2 done  (79s)
r= 4 done  (117s)
r= 8 done  (155s)
r=16 done  (194s)
r=30 done  (232s)

RAW

RECOVERY

d      1      2      4      8      16     30
r                                           
1   0.998  0.998  0.995  0.991  0.986  0.982
2   0.785  0.997  0.993  0.986  0.978  0.975
4   0.666  0.841  0.996  0.991  0.984  0.981
8   0.536  0.709  0.886  0.992  0.984  0.982
16  0.468  0.609  0.775  0.919  0.979  0.975
30  0.391  0.554  0.718  0.870  0.962  0.965

TEST NLL

d      1      2      4      8      16     30
r                                           
1   0.901  0.905  0.909  0.916  0.924  0.926
2   1.070  0.835  0.840  0.845  0.852  0.855
4   1.321  1.131  0.918  0.930  0.942  0.944
8   1.430  1.296  1.089  0.946  0.957  0.960
16  1.365  1.262  1.111  0.951  0.879  0.882
30  1.422  1.330  1.185  1.010  0.898  0.891

SOFTMAX

RECOVERY

d      1      2      4      8      16     30
r                                           
1   0.007  0.262  0.137  0.305  0.146

## Cell 6: Behavioural recovery probe

The weight-based metric reads $q_i \cdot k_j$ directly. That is valid for the
raw model, where $q_i \cdot k_j$ *is* the pull, but not for the softmax model,
where the effective pull is $\alpha_{ij} w_j$ — spread across the attention
weights and the value pathway. A near-zero weight-based score there is
therefore ambiguous: the structure may be absent, or merely not visible in
$q \cdot k$.

**Probe.** Estimate the pull behaviourally, using only forward passes:

1. Fix a target item $i$ and a probe item $j$.
2. Draw a random filler set $F$ of size $M-2$ from the remaining items.
3. Score $i$ in the slate $\{i, j\} \cup F$, and in $\{i, j'\} \cup F$ for
   every other candidate $j'$ — same fillers, only the probe slot swapped.
4. The spread of $i$'s score across probe items estimates $\hat{C}_{ij}$ up to
   a per-row constant, which centering removes.
5. Average over many filler draws to reduce variance.

This treats the model as a black box and works for any architecture, including
PASAR. Validation: on the raw model it must reproduce the weight-based number.
If it does not, the probe is wrong, not the model.

In [ ]:
@torch.no_grad()
def behavioural_C(model, N, set_size, n_fillers=40, seed=0, device=None):
    """Estimate effective pull C[i,j] from forward passes only."""
    dev = device or next(model.parameters()).device
    model.eval()
    g = np.random.default_rng(seed)
    C_hat = np.zeros((N, N))

    for i in range(N):
        others = np.setdiff1d(np.arange(N), [i])
        acc = np.zeros(N)
        for _ in range(n_fillers):
            # fillers exclude i; probe j is drawn from what remains
            F = g.choice(others, size=set_size - 2, replace=False)
            cand = np.setdiff1d(others, F)                    # valid probes
            slates = np.stack([np.concatenate(([i], [j], F)) for j in cand])
            logits = model(torch.tensor(slates, dtype=torch.long, device=dev))
            acc[cand] += logits[:, 0].cpu().numpy()           # score of i (slot 0)
        counts = np.zeros(N)
        for _ in range(n_fillers):
            pass
        # normalise by how often each j was a valid probe (approx uniform)
        C_hat[i] = acc / n_fillers * (N - 1) / max(len(cand), 1)
    return C_hat


print("Validating probe on the raw model (should match weight-based)\n")
C_v = make_C(N, 4, scale=0.8, seed=0)
u_v = make_utilities(N, scale=1.0, seed=0)
X_v, y_v = generate_choices(N, C_v, u_v, n_samples, set_size, seed=0)

m_raw, _, _ = train_eval(ItemAttention, X_v, y_v, N, d=4, seed=0)
w_raw = recovery_score(C_v, m_raw.implied_C())
b_raw = recovery_score(C_v, behavioural_C(m_raw, N, set_size, seed=0))
print(f"  raw   weight-based {w_raw:.3f}   behavioural {b_raw:.3f}")

m_sm, _, _ = train_eval(ItemAttentionSoftmax, X_v, y_v, N, d=4, seed=0)
w_sm = recovery_score(C_v, m_sm.implied_C())
b_sm = recovery_score(C_v, behavioural_C(m_sm, N, set_size, seed=0))
print(f"  sm    weight-based {w_sm:.3f}   behavioural {b_sm:.3f}")

Validating probe on the raw model (should match weight-based)

  raw   weight-based 0.993   behavioural 0.961
  sm    weight-based -0.038   behavioural 0.737


## Cell 7: Corrected probe + softmax rank sweep

**Probe fix.** The previous version divided every accumulator by `n_fillers`,
but each filler draw removes a different subset from the valid probe pool, so
items were sampled unequally. We now track per-item counts and divide by the
actual count. This removes the noise that attenuated the raw model's score
(0.993 → 0.961), so both models' numbers should tighten.

**Sweep.** With a readout that is valid for both architectures, we rerun the
full $r \times d$ grid for softmax. The question: does the $d = r$ boundary
seen in the raw model reappear?

- Boundary present → the rank ceiling is a property of attention generally.
- Boundary absent → the ceiling is specific to additive scoring, and softmax
  is constrained by something else.

Fewer seeds (2) and fillers (30) here, since the probe requires many forward
passes per cell.

In [ ]:
@torch.no_grad()
def behavioural_C(model, N, set_size, n_fillers=40, seed=0, device=None):
    """Estimate effective pull C[i,j] from forward passes only. Count-corrected."""
    dev = device or next(model.parameters()).device
    model.eval()
    g = np.random.default_rng(seed)
    C_hat = np.zeros((N, N))

    for i in range(N):
        others = np.setdiff1d(np.arange(N), [i])
        acc    = np.zeros(N)
        counts = np.zeros(N)
        for _ in range(n_fillers):
            F = g.choice(others, size=set_size - 2, replace=False)
            cand = np.setdiff1d(others, F)
            slates = np.stack([np.concatenate(([i], [j], F)) for j in cand])
            logits = model(torch.tensor(slates, dtype=torch.long, device=dev))
            acc[cand]    += logits[:, 0].cpu().numpy()
            counts[cand] += 1
        C_hat[i] = np.divide(acc, counts, out=np.zeros(N), where=counts > 0)
    return C_hat


# --- re-validate with the fix ---
print("Re-validation (corrected probe)\n")
for nm, mdl in [('raw', m_raw), ('softmax', m_sm)]:
    b = recovery_score(C_v, behavioural_C(mdl, N, set_size, n_fillers=40, seed=0))
    print(f"  {nm:8s} behavioural {b:.3f}")

# --- softmax rank sweep, behavioural readout ---
rows3 = []
t0 = time.time()
for r_true in r_values:
    for s_ in [0, 1]:
        C_s = make_C(N, r_true, scale=0.8, seed=100 * s_ + r_true)
        u_s = make_utilities(N, scale=1.0, seed=100 * s_ + r_true)
        X, y = generate_choices(N, C_s, u_s, n_samples, set_size,
                                seed=100 * s_ + r_true)
        for d_head in d_values:
            for name, cls in [('raw', ItemAttention),
                              ('softmax', ItemAttentionSoftmax)]:
                m, _, te = train_eval(cls, X, y, N, d_head, seed=s_)
                rows3.append({
                    'model': name, 'r': r_true, 'd': d_head, 'seed': s_,
                    'beh': recovery_score(C_s, behavioural_C(m, N, set_size,
                                                             n_fillers=30, seed=s_)),
                    'wt':  recovery_score(C_s, m.implied_C()),
                    'test_nll': te})
    print(f"r={r_true:2d} done  ({time.time()-t0:.0f}s)")

df3 = pd.DataFrame(rows3)
df3.to_csv('exp4_behavioural_sweep.csv', index=False)

for name in ['raw', 'softmax']:
    sub = df3[df3.model == name]
    print(f"\n{'='*52}\n{name.upper()}  — behavioural recovery\n{'='*52}\n")
    print(sub.pivot_table(index='r', columns='d', values='beh',
                          aggfunc='mean').round(3).to_string())

print(f"\n{'='*52}\nSOFTMAX: behavioural minus weight readout\n{'='*52}\n")
sm = df3[df3.model == 'softmax']
gap = (sm.pivot_table(index='r', columns='d', values='beh', aggfunc='mean')
       - sm.pivot_table(index='r', columns='d', values='wt', aggfunc='mean'))
print(gap.round(3).to_string())

Re-validation (corrected probe)

  raw      behavioural 0.977
  softmax  behavioural 0.808
r= 1 done  (43s)
r= 2 done  (85s)
r= 4 done  (126s)
r= 8 done  (168s)
r=16 done  (210s)
r=30 done  (252s)

RAW  — behavioural recovery

d      1      2      4      8      16     30
r                                           
1   0.985  0.984  0.982  0.978  0.973  0.968
2   0.715  0.979  0.975  0.970  0.962  0.960
4   0.632  0.824  0.981  0.975  0.969  0.964
8   0.523  0.696  0.876  0.976  0.967  0.964
16  0.435  0.580  0.753  0.903  0.960  0.957
30  0.396  0.545  0.714  0.860  0.947  0.946

SOFTMAX  — behavioural recovery

d      1      2      4      8      16     30
r                                           
1   0.822  0.827  0.842  0.810  0.830  0.780
2   0.691  0.808  0.819  0.821  0.798  0.796
4   0.547  0.681  0.783  0.791  0.808  0.816
8   0.522  0.621  0.686  0.763  0.788  0.787
16  0.444  0.513  0.664  0.747  0.789  0.801
30  0.361  0.507  0.574  0.696  0.751  0.750

SOFTMAX: behaviour